# CDN on MovieLens-1M — Cross Decoupling Network

## 0. Config and imports

In [1]:
import gc
import itertools
import json
import re
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm


def set_seed(seed: int = 0):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


CFG = {
    "work_dir": "../data",

    "embed_dim": 64,
    "tower_hidden": [256, 128, 64],
    "expert_hidden": 64,
    "max_title_words": 2000,

    "batch_size": 1024,
    "num_workers": 0,
    "grad_clip": 1.0,

    "cdn_tune_epochs": 3,
    "final_epochs": 40,
    "temperature": 1.0,

    "skip_tune_if_cached": True,
    "cache_path": "movielens_cdn_best_hparams.json",

    "full_grid": True,
    "tune_val_fraction": 1.0,

    "seeds": [0, 1, 2, 3, 4],

    "eval_k": [10, 50],
    "eval_batch_users": 128,
    "eval_item_batch": 1024,
    "head_fraction": 0.20,

    "seed": 0,
}

set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print(f"Final: {len(CFG['seeds'])} seeds × {CFG['final_epochs']} epochs")


device: cpu
Final: 5 seeds × 40 epochs


## 1. Load MovieLens-1M

In [2]:
GENRES = [
    "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical",
    "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western",
]


def read_ml1m(raw: Path):
    users = pd.read_csv(
        raw / "users.dat",
        sep="::",
        engine="python",
        names=["user_id", "gender", "age", "occupation", "zip"],
        encoding="latin-1",
    )

    ratings = pd.read_csv(
        raw / "ratings.dat",
        sep="::",
        engine="python",
        names=["user_id", "item_id", "rating", "timestamp"],
        encoding="latin-1",
    )
    ratings = ratings[ratings["rating"] > 0].copy()

    movies = []
    with open(raw / "movies.dat", encoding="latin-1") as f:
        for line in f:
            parts = line.strip().split("::")
            mid = int(parts[0])
            title = parts[1]
            genres = parts[2].split("|") if len(parts) > 2 else []
            year = 1995
            if title.endswith(")") and "(" in title:
                try:
                    year = int(title[title.rfind("(") + 1 : -1])
                except ValueError:
                    pass
            movies.append({"item_id": mid, "title": title, "genres": genres, "year": year})

    return users, pd.DataFrame(movies), ratings


RAW_DIR = Path(CFG["work_dir"])
users_df, movies_df, ratings = read_ml1m(RAW_DIR)

item_ids = sorted(ratings["item_id"].unique())
uid_map = {u: i for i, u in enumerate(users_df["user_id"].tolist())}
iid_map = {m: i for i, m in enumerate(item_ids)}

NUM_USERS = len(uid_map)
NUM_ITEMS = len(iid_map)

print(f"users={NUM_USERS:,} items={NUM_ITEMS:,} ratings={len(ratings):,}")


users=6,040 items=3,706 ratings=1,000,209


## 2. Feature engineering

In [3]:
def tokenize_title(title: str):
    if "(" in title:
        title = title.rsplit("(", 1)[0]
    return re.findall(r"[a-z0-9]+", title.lower())


def build_title_bow(titles, max_words: int):
    counter = Counter()
    per_title = []
    for title in titles:
        toks = tokenize_title(title)
        per_title.append(toks)
        counter.update(toks)

    vocab = [w for w, _ in counter.most_common(max_words)]
    w2i = {w: i for i, w in enumerate(vocab)}

    mat = np.zeros((len(titles), len(vocab)), dtype=np.float32)
    for r, toks in enumerate(per_title):
        for tok in toks:
            j = w2i.get(tok)
            if j is not None:
                mat[r, j] = 1.0
    return mat


# ---------- item features ----------
genre_to_idx = {g: i for i, g in enumerate(GENRES)}
genre_matrix, years, titles = [], [], []

for mid in item_ids:
    row = movies_df.loc[movies_df["item_id"] == mid].iloc[0]
    gvec = np.zeros(len(GENRES), dtype=np.float32)
    for g in row["genres"]:
        if g in genre_to_idx:
            gvec[genre_to_idx[g]] = 1.0
    genre_matrix.append(gvec)
    years.append(row["year"])
    titles.append(row["title"])

years = np.asarray(years, dtype=np.float32)
years = (years - years.mean()) / (years.std() + 1e-6)
title_bow = build_title_bow(titles, CFG["max_title_words"])

ITEM_GEN = np.hstack([
    np.stack(genre_matrix),
    years.reshape(-1, 1),
    title_bow,
]).astype(np.float32)

ITEM_GEN = (ITEM_GEN - ITEM_GEN.mean(axis=0, keepdims=True)) / (ITEM_GEN.std(axis=0, keepdims=True) + 1e-6)
ITEM_GEN = ITEM_GEN.astype(np.float32)
ITEM_GEN_DIM = ITEM_GEN.shape[1]


# ---------- user features ----------
gender_map = {g: i for i, g in enumerate(sorted(users_df["gender"].unique()))}
age_map = {a: i for i, a in enumerate(sorted(users_df["age"].unique()))}
occupation_map = {o: i for i, o in enumerate(sorted(users_df["occupation"].unique()))}
zip_map = {z: i for i, z in enumerate(sorted(users_df["zip"].unique()))}

USER_GEN = np.stack([
    users_df["gender"].map(gender_map),
    users_df["age"].map(age_map),
    users_df["occupation"].map(occupation_map),
    users_df["zip"].map(zip_map),
], axis=1).astype(np.int64)

USER_GEN_SIZES = [len(gender_map), len(age_map), len(occupation_map), len(zip_map)]

print("ITEM_GEN_DIM:", ITEM_GEN_DIM)
print("USER_GEN_SIZES:", USER_GEN_SIZES)


ITEM_GEN_DIM: 2019
USER_GEN_SIZES: [2, 7, 21, 3439]


## 3. Leave-one-out split

In [4]:
train_pairs = []
val_u, val_i = [], []
test_u, test_i = [], []
train_user_items = [set() for _ in range(NUM_USERS)]

for uid, grp in ratings.groupby("user_id"):
    if uid not in uid_map:
        continue
    u = uid_map[uid]
    item_seq = [iid_map[i] for i in grp.sort_values("timestamp")["item_id"].tolist() if i in iid_map]
    if len(item_seq) < 3:
        continue
    for it in item_seq[:-2]:
        train_pairs.append((u, it))
        train_user_items[u].add(it)
    val_u.append(u)
    val_i.append(item_seq[-2])
    test_u.append(u)
    test_i.append(item_seq[-1])

train_pairs = np.asarray(train_pairs, dtype=np.int64)
val_u = np.asarray(val_u, dtype=np.int64)
val_i = np.asarray(val_i, dtype=np.int64)
test_u = np.asarray(test_u, dtype=np.int64)
test_i = np.asarray(test_i, dtype=np.int64)

print(f"train={len(train_pairs):,} val={len(val_u):,} test={len(test_u):,}")

known_val = [set(s) for s in train_user_items]
known_test = [set(s) for s in train_user_items]
for u, i in zip(val_u, val_i):
    known_test[int(u)].add(int(i))

item_freq = np.bincount(train_pairs[:, 1], minlength=NUM_ITEMS)
order = np.argsort(-item_freq)
n_head = max(1, int(NUM_ITEMS * CFG["head_fraction"]))
head_mask = np.zeros(NUM_ITEMS, dtype=bool)
head_mask[order[:n_head]] = True

print(f"head={head_mask.sum():,} tail={(~head_mask).sum():,}")
print(f"test_head_share={head_mask[test_i].mean():.4f}")
print(f"test_tail_share={(~head_mask[test_i]).mean():.4f}")


train=988,129 val=6,040 test=6,040
head=741 tail=2,965
test_head_share=0.6088
test_tail_share=0.3912


## 4. Rebalanced distribution $\Omega_r$

In [6]:
def make_regularizer_pairs(seed: int = 0):
    """
    Build Omega_r:
    - keep all tail feedback;
    - downsample head feedback to max frequency of tail items.
    """
    rng = np.random.default_rng(seed)
    pairs_by_item = defaultdict(list)
    for u, i in train_pairs:
        pairs_by_item[int(i)].append((int(u), int(i)))

    tail_freq = item_freq[~head_mask]
    tail_freq = tail_freq[tail_freq > 0]
    max_tail_freq = int(tail_freq.max()) if len(tail_freq) else 1

    reg_pairs = []
    for i in range(NUM_ITEMS):
        pairs = pairs_by_item.get(i, [])
        if not pairs:
            continue
        if head_mask[i] and len(pairs) > max_tail_freq:
            idx = rng.choice(len(pairs), size=max_tail_freq, replace=False)
            reg_pairs.extend([pairs[j] for j in idx])
        else:
            reg_pairs.extend(pairs)

    rng.shuffle(reg_pairs)
    return np.asarray(reg_pairs, dtype=np.int64), max_tail_freq


reg_pairs, max_tail_freq = make_regularizer_pairs(seed=CFG["seed"])
print(f"Omega_m train pairs={len(train_pairs):,}")
print(f"Omega_r regularizer pairs={len(reg_pairs):,}")
print(f"max_tail_freq={max_tail_freq:,}")
print(f"Omega_r ratio={len(reg_pairs) / max(len(train_pairs), 1):.3f}")


Omega_m train pairs=988,129
Omega_r regularizer pairs=657,171
max_tail_freq=423
Omega_r ratio=0.665


## 5. Dataset and CDN model

In [7]:
class PairDataset(Dataset):
    def __init__(self, pairs: np.ndarray):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u, i = self.pairs[idx]
        return int(u), int(i)


def make_loader(pairs: np.ndarray, batch_size: int, shuffle: bool = True):
    return DataLoader(
        PairDataset(pairs),
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=True,
        num_workers=CFG["num_workers"],
        pin_memory=torch.cuda.is_available(),
    )


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: list[int], dropout: float = 0.0):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden:
            layers.append(nn.Linear(d, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d = h
        self.net = nn.Sequential(*layers)
        self.out_dim = d

    def forward(self, x):
        return self.net(x)


class UserGenEnc(nn.Module):
    def __init__(self, sizes: list[int], dim: int = 16):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(n, dim) for n in sizes])
        self.out_dim = len(sizes) * dim

    def forward(self, x):
        return torch.cat([emb(x[:, i].long()) for i, emb in enumerate(self.embs)], dim=-1)


class CDN(nn.Module):
    def __init__(self, dropout: float = 0.0, n_mem_experts: int = 1, n_gen_experts: int = 1):
        super().__init__()
        ed = CFG["embed_dim"]
        h = CFG["tower_hidden"]
        expert_hidden = CFG["expert_hidden"]

        self.n_mem_experts = n_mem_experts
        self.n_gen_experts = n_gen_experts
        self.out_dim = h[-1]

        # User side: shared part + main/regularizer branches
        self.user_id = nn.Embedding(NUM_USERS, ed)
        self.user_gen = UserGenEnc(USER_GEN_SIZES, dim=16)
        user_in = ed + self.user_gen.out_dim
        self.user_shared = MLP(user_in, [h[0]], dropout)
        self.user_main = MLP(h[0], h[1:], dropout)
        self.user_reg = MLP(h[0], h[1:], dropout)
        self.user_main_ln = nn.LayerNorm(self.out_dim)
        self.user_reg_ln = nn.LayerNorm(self.out_dim)

        # Item side MoE
        self.item_id = nn.Embedding(NUM_ITEMS, ed)
        self.mem_experts = nn.ModuleList([MLP(ed, [expert_hidden], dropout) for _ in range(n_mem_experts)])
        self.gen_experts = nn.ModuleList([MLP(ITEM_GEN_DIM, [expert_hidden], dropout) for _ in range(n_gen_experts)])
        self.mem_proj = nn.Linear(expert_hidden, self.out_dim)
        self.gen_proj = nn.Linear(expert_hidden, self.out_dim)
        self.gate = nn.Linear(1, n_mem_experts + n_gen_experts)
        self.item_ln = nn.LayerNorm(self.out_dim)

        self.init_weights()

    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def user_base(self, u, ug):
        x = torch.cat([self.user_id(u), self.user_gen(ug)], dim=-1)
        return self.user_shared(x)

    def user_vec_main(self, u, ug):
        x = self.user_base(u, ug)
        return self.user_main_ln(self.user_main(x))

    def user_vec_reg(self, u, ug):
        x = self.user_base(u, ug)
        return self.user_reg_ln(self.user_reg(x))

    def item_vec_moe(self, i, ig):
        id_x = self.item_id(i)
        mem = torch.stack([self.mem_proj(expert(id_x)) for expert in self.mem_experts], dim=1)
        gen = torch.stack([self.gen_proj(expert(ig)) for expert in self.gen_experts], dim=1)
        experts = torch.cat([mem, gen], dim=1)
        gate_in = item_pop_t[i].unsqueeze(-1)
        weights = torch.softmax(self.gate(gate_in), dim=-1).unsqueeze(-1)
        y = (experts * weights).sum(dim=1)
        return self.item_ln(y)

    # Inference uses main branch only.
    def user_vec(self, u, ug):
        return self.user_vec_main(u, ug)

    def item_vec(self, i, ig):
        return self.item_vec_moe(i, ig)

    def cdn_losses(self, main_users, main_items, reg_users, reg_items, alpha: float):
        """
        Correct CDN objective for in-batch softmax.

        We do NOT mix logits_m and logits_r if main_items and reg_items are different.
        Instead:
            loss = alpha * CE(logits_m) + (1-alpha) * CE(logits_r)
        """
        xm = F.normalize(self.user_vec_main(main_users, ug_t[main_users]), dim=-1, eps=1e-6)
        ym = F.normalize(self.item_vec_moe(main_items, ig_t[main_items]), dim=-1, eps=1e-6)
        xr = F.normalize(self.user_vec_reg(reg_users, ug_t[reg_users]), dim=-1, eps=1e-6)
        yr = F.normalize(self.item_vec_moe(reg_items, ig_t[reg_items]), dim=-1, eps=1e-6)

        logits_m = (xm @ ym.T) / CFG.get("temperature", 1.0)
        logits_r = (xr @ yr.T) / CFG.get("temperature", 1.0)
        logits_m = torch.clamp(logits_m, -20.0, 20.0)
        logits_r = torch.clamp(logits_r, -20.0, 20.0)

        if not torch.isfinite(logits_m).all():
            raise RuntimeError("NaN/Inf in main logits")
        if not torch.isfinite(logits_r).all():
            raise RuntimeError("NaN/Inf in regularizer logits")

        labels_m = torch.arange(logits_m.size(0), device=logits_m.device)
        labels_r = torch.arange(logits_r.size(0), device=logits_r.device)

        loss_m = F.cross_entropy(logits_m, labels_m)
        loss_r = F.cross_entropy(logits_r, labels_r)
        loss = alpha * loss_m + (1.0 - alpha) * loss_r
        return loss, loss_m, loss_r


ug_t = torch.from_numpy(USER_GEN).long().to(device)
ig_t = torch.from_numpy(ITEM_GEN).float().to(device)

item_pop = np.log1p(item_freq.astype(np.float32))
item_pop = (item_pop - item_pop.mean()) / (item_pop.std() + 1e-6)
item_pop_t = torch.from_numpy(item_pop.astype(np.float32)).to(device)

main_loader = make_loader(train_pairs, CFG["batch_size"], shuffle=True)
reg_loader = make_loader(reg_pairs, CFG["batch_size"], shuffle=True)

print("CDN helpers ready")
print("main batches:", len(main_loader))
print("reg batches:", len(reg_loader))


CDN helpers ready
main batches: 964
reg batches: 641


## 6. Evaluation

In [8]:
@torch.no_grad()
def evaluate_full_corpus(
    model: nn.Module,
    users: np.ndarray,
    true_items: np.ndarray,
    known_user_items: List[set],
    head_mask: np.ndarray,
    ks: List[int],
    item_freq: np.ndarray,
    user_batch_size: int = 128,
    item_batch_size: int = 1024,
):
    model.eval()
    ranks_all, ranks_head, ranks_tail = [], [], []
    max_k = max(ks)
    recommended_by_k = {k: [] for k in ks}

    item_vectors = []
    for s in tqdm(range(0, NUM_ITEMS, item_batch_size), desc="item vectors", leave=False):
        e = min(s + item_batch_size, NUM_ITEMS)
        idx = torch.arange(s, e, device=device)
        v = model.item_vec(idx, ig_t[idx])
        v = F.normalize(v, dim=-1, eps=1e-6)
        if not torch.isfinite(v).all():
            raise RuntimeError(f"Non-finite item vectors on item batch {s}:{e}")
        item_vectors.append(v.cpu())
    item_vectors = torch.cat(item_vectors, dim=0).to(device)

    for start in tqdm(range(0, len(users), user_batch_size), desc="eval users", leave=False):
        end = min(start + user_batch_size, len(users))
        bu = users[start:end]
        bi = true_items[start:end]
        ut = torch.tensor(bu, device=device, dtype=torch.long)
        uvec = model.user_vec(ut, ug_t[ut])
        uvec = F.normalize(uvec, dim=-1, eps=1e-6)
        if not torch.isfinite(uvec).all():
            raise RuntimeError(f"Non-finite user vectors on user batch {start}:{end}")

        scores = (uvec @ item_vectors.T).cpu().numpy()
        if not np.isfinite(scores).all():
            bad = np.sum(~np.isfinite(scores))
            raise RuntimeError(f"Non-finite scores in user batch {start}:{end}: {bad}")

        for row, (u, true_i) in enumerate(zip(bu, bi)):
            u = int(u)
            true_i = int(true_i)
            srow = scores[row].copy()

            for it in known_user_items[u]:
                if it != true_i:
                    srow[it] = -1e9

            true_score = srow[true_i]
            eps = 1e-12
            num_greater = int((srow > true_score + eps).sum())
            num_tied = int((np.abs(srow - true_score) <= eps).sum())
            rank = num_greater + num_tied - 1

            ranks_all.append(rank)
            if head_mask[true_i]:
                ranks_head.append(rank)
            else:
                ranks_tail.append(rank)

            if max_k < len(srow):
                top_unsorted = np.argpartition(-srow, max_k - 1)[:max_k]
                top_idx = top_unsorted[np.argsort(-srow[top_unsorted])]
            else:
                top_idx = np.argsort(-srow)

            for k in ks:
                recommended_by_k[k].append(top_idx[:k].astype(np.int64))

    def agg_accuracy(ranks: List[int]) -> Dict[str, float]:
        a = np.asarray(ranks, dtype=np.int64)
        out = {}
        for k in ks:
            if len(a) == 0:
                out[f"HR@{k}"] = np.nan
                out[f"NDCG@{k}"] = np.nan
            else:
                hits = a < k
                out[f"HR@{k}"] = 100.0 * hits.mean()
                out[f"NDCG@{k}"] = 100.0 * np.where(hits, 1.0 / np.log2(a + 2), 0.0).mean()
        return out

    tail_mask = ~head_mask
    num_tail_items = int(tail_mask.sum())
    popularity = np.log1p(item_freq.astype(np.float64))
    long_tail = {}

    for k in ks:
        recs = np.vstack(recommended_by_k[k])
        unique_recs = np.unique(recs)
        catalog_coverage = len(unique_recs) / NUM_ITEMS
        tail_coverage = np.sum(tail_mask[unique_recs]) / num_tail_items if num_tail_items > 0 else np.nan
        avg_popularity = popularity[recs].mean()
        tail_ratio = tail_mask[recs].mean()

        n_users_eval = recs.shape[0]
        if n_users_eval <= 1:
            personalization = np.nan
        else:
            exposure_counts = np.bincount(recs.reshape(-1), minlength=NUM_ITEMS)
            pairwise_intersections = np.sum(exposure_counts * (exposure_counts - 1) / 2)
            num_user_pairs = n_users_eval * (n_users_eval - 1) / 2
            avg_overlap = pairwise_intersections / num_user_pairs
            personalization = 1.0 - avg_overlap / k

        long_tail[f"CatalogCoverage@{k}"] = 100.0 * catalog_coverage
        long_tail[f"TailCoverage@{k}"] = 100.0 * tail_coverage
        long_tail[f"AvgPopularity@{k}"] = float(avg_popularity)
        long_tail[f"TailRatio@{k}"] = 100.0 * tail_ratio
        long_tail[f"Personalization@{k}"] = 100.0 * personalization

    return {
        "overall": agg_accuracy(ranks_all),
        "head": agg_accuracy(ranks_head),
        "tail": agg_accuracy(ranks_tail),
        "long_tail": long_tail,
        "counts": {
            "overall": len(ranks_all),
            "head": len(ranks_head),
            "tail": len(ranks_tail),
        },
    }


def print_metrics(metrics: Dict):
    print("counts:", metrics.get("counts", {}))
    for split in ("overall", "head", "tail"):
        print(f"[{split}]", metrics[split])
    if "long_tail" in metrics:
        print("[long_tail]", metrics["long_tail"])


## 7. CDN training helpers

In [9]:
def alpha_for_epoch(epoch: int, total_epochs: int, gamma: float):
    return 1.0 - (epoch / (gamma * total_epochs)) ** 2


def train_one_cdn_epoch(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    main_loader: DataLoader,
    reg_loader: DataLoader,
    epoch: int,
    total_epochs: int,
    gamma: float,
    desc: str,
):
    model.train()
    alpha = alpha_for_epoch(epoch, total_epochs, gamma)
    total_loss = 0.0
    total_n = 0
    reg_iter = iter(reg_loader)

    pbar = tqdm(main_loader, desc=f"{desc} {epoch}/{total_epochs} alpha={alpha:.3f}", leave=False)
    for main_users, main_items in pbar:
        try:
            reg_users, reg_items = next(reg_iter)
        except StopIteration:
            reg_iter = iter(reg_loader)
            reg_users, reg_items = next(reg_iter)

        main_users = main_users.to(device, non_blocking=True)
        main_items = main_items.to(device, non_blocking=True)
        reg_users = reg_users.to(device, non_blocking=True)
        reg_items = reg_items.to(device, non_blocking=True)

        loss, loss_m, loss_r = model.cdn_losses(
            main_users=main_users,
            main_items=main_items,
            reg_users=reg_users,
            reg_items=reg_items,
            alpha=alpha,
        )

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite CDN loss at epoch={epoch}: {loss.item()}")

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        optimizer.step()

        bs = main_users.size(0)
        total_loss += loss.item() * bs
        total_n += bs
        pbar.set_postfix(loss=f"{loss.item():.4f}", loss_m=f"{loss_m.item():.4f}", loss_r=f"{loss_r.item():.4f}")

    return total_loss / max(total_n, 1)


def train_cdn(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    epochs: int,
    gamma: float,
    main_loader: DataLoader,
    reg_loader: DataLoader,
    val_u: np.ndarray,
    val_i: np.ndarray,
    known_val: List[set],
    head_mask: np.ndarray,
    item_freq: np.ndarray,
    seed_tag: str = "",
    track_val: bool = True,
):
    best_val_hr50 = -1.0
    best_state = None
    history = []
    k_track = CFG["eval_k"][-1]

    for ep in range(1, epochs + 1):
        avg_loss = train_one_cdn_epoch(
            model=model,
            optimizer=optimizer,
            main_loader=main_loader,
            reg_loader=reg_loader,
            epoch=ep,
            total_epochs=epochs,
            gamma=gamma,
            desc=f"CDN {seed_tag}",
        )

        log = {"epoch": ep, "alpha": alpha_for_epoch(ep, epochs, gamma), "train_loss": avg_loss}

        if track_val:
            vm = evaluate_full_corpus(
                model=model,
                users=val_u,
                true_items=val_i,
                known_user_items=known_val,
                head_mask=head_mask,
                ks=CFG["eval_k"],
                item_freq=item_freq,
                user_batch_size=CFG["eval_batch_users"],
                item_batch_size=CFG["eval_item_batch"],
            )
            hr = vm["overall"].get(f"HR@{k_track}", -1.0)
            log["val_HR@50"] = hr

            if hr > best_val_hr50:
                best_val_hr50 = hr
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

            print(f"  CDN ep{ep} alpha={log['alpha']:.4f} loss={avg_loss:.4f} val HR@{k_track}={hr:.4f}")
        else:
            print(f"  CDN ep{ep} alpha={log['alpha']:.4f} loss={avg_loss:.4f}")

        history.append(log)

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return best_state, best_val_hr50, history


## 8. Grid search

In [ ]:
LR_GRID = [0.1, 0.01, 0.001, 0.0001]
DROPOUT_GRID = [0.1, 0.3, 0.5, 0.7, 0.9]
WD_GRID = [0.0, 0.1, 0.01, 0.001]

GAMMA_GRID = [1.5, 3.0, 10.0]
EXPERT_COUNT_GRID = [1, 2]

if not CFG["full_grid"]:
    LR_GRID = [0.01, 0.001]
    DROPOUT_GRID = [0.1, 0.3]
    WD_GRID = [0.0, 0.001]
    GAMMA_GRID = [1.5, 3.0, 10.0]
    EXPERT_COUNT_GRID = [1, 2]

combos = list(itertools.product(LR_GRID, DROPOUT_GRID, WD_GRID, GAMMA_GRID, EXPERT_COUNT_GRID))
k_main = CFG["eval_k"][-1]
cache_path = Path(CFG["cache_path"])

if CFG["skip_tune_if_cached"] and cache_path.exists():
    best_hp = json.loads(cache_path.read_text())
    print(f"Loaded cached hparams: {best_hp}")

else:
    frac = float(CFG.get("tune_val_fraction", 1.0))
    if frac < 1.0:
        n_tune = max(1, int(len(val_u) * frac))
        idx = np.random.default_rng(42).choice(len(val_u), n_tune, replace=False)
        val_u_t, val_i_t = val_u[idx], val_i[idx]
    else:
        val_u_t, val_i_t = val_u, val_i

    tune_ep = CFG["cdn_tune_epochs"]
    print(f"Tuning CDN {len(combos)} trials × {tune_ep} ep (val subset: {len(val_u_t):,}/{len(val_u):,})")

    best_hr = -1.0
    best_hp = None

    for ti, (lr, dr, wd, gamma, n_experts) in enumerate(combos, 1):
        set_seed(CFG["seed"])
        m = CDN(dropout=dr, n_mem_experts=n_experts, n_gen_experts=n_experts).to(device)
        opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=wd)
        status = "ok"
        hr = -1.0

        try:
            _, hr, _ = train_cdn(
                model=m,
                optimizer=opt,
                epochs=tune_ep,
                gamma=gamma,
                main_loader=main_loader,
                reg_loader=reg_loader,
                val_u=val_u_t,
                val_i=val_i_t,
                known_val=known_val,
                head_mask=head_mask,
                item_freq=item_freq,
                seed_tag=f"trial{ti}",
                track_val=True,
            )
            if not np.isfinite(hr):
                raise RuntimeError(f"Non-finite HR: {hr}")
        except RuntimeError as e:
            status = "failed"
            print(f"  CDN trial {ti} FAILED: {e}")

        print(
            f"  CDN trial {ti:4d}/{len(combos)} "
            f"lr={lr:.0e} dr={dr} wd={wd:.0e} gamma={gamma:g} experts={n_experts} "
            f"-> val HR@{k_main}={hr:.2f}%"
        )

        if status == "ok" and hr > best_hr:
            best_hr = hr
            best_hp = {
                "lr": lr,
                "dropout": dr,
                "weight_decay": wd,
                "gamma": gamma,
                "n_experts": n_experts,
                f"val_HR@{k_main}": hr,
            }

        del m, opt
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if best_hp is None:
        raise RuntimeError("No successful grid-search trial.")

    cache_path.write_text(json.dumps(best_hp, indent=2))
    print(f"\nBest val HR@{k_main}={best_hr:.2f}% -> {best_hp}")
    print(f"Saved best hparams: {cache_path}")


Tuning CDN 1680 trials × 3 ep (val subset: 6,040/6,040)


KeyboardInterrupt: 

## 9. Final training over 5 seeds

In [ ]:
all_rows = []
all_test = []

print("Using best_hp:", best_hp)

for seed in CFG["seeds"]:
    print("\n" + "=" * 80)
    print(f"MovieLens CDN seed {seed}")
    print("=" * 80)

    set_seed(seed)

    # Rebuild Omega_r per seed so downsampling randomness is seed-specific.
    reg_pairs_seed, max_tail_freq_seed = make_regularizer_pairs(seed=seed)
    main_loader_seed = make_loader(train_pairs, CFG["batch_size"], shuffle=True)
    reg_loader_seed = make_loader(reg_pairs_seed, CFG["batch_size"], shuffle=True)

    print(f"Omega_r seed={seed}: {len(reg_pairs_seed):,} pairs, max_tail_freq={max_tail_freq_seed:,}")

    model = CDN(
        dropout=best_hp["dropout"],
        n_mem_experts=int(best_hp["n_experts"]),
        n_gen_experts=int(best_hp["n_experts"]),
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=best_hp["lr"], weight_decay=best_hp["weight_decay"])

    best_state, best_val_hr50, _ = train_cdn(
        model=model,
        optimizer=optimizer,
        epochs=CFG["final_epochs"],
        gamma=float(best_hp["gamma"]),
        main_loader=main_loader_seed,
        reg_loader=reg_loader_seed,
        val_u=val_u,
        val_i=val_i,
        known_val=known_val,
        head_mask=head_mask,
        item_freq=item_freq,
        seed_tag=f"seed{seed}",
        track_val=True,
    )

    print(f"seed {seed} best val HR@50: {best_val_hr50:.4f}")

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    test_metrics = evaluate_full_corpus(
        model=model,
        users=test_u,
        true_items=test_i,
        known_user_items=known_test,
        head_mask=head_mask,
        ks=CFG["eval_k"],
        item_freq=item_freq,
        user_batch_size=CFG["eval_batch_users"],
        item_batch_size=CFG["eval_item_batch"],
    )

    print("TEST METRICS")
    print_metrics(test_metrics)
    all_test.append(test_metrics)

    row = {
        "method": "CDN",
        "seed": seed,
        "lr": best_hp["lr"],
        "dropout": best_hp["dropout"],
        "weight_decay": best_hp["weight_decay"],
        "gamma": best_hp["gamma"],
        "n_experts": best_hp["n_experts"],
        "final_epochs": CFG["final_epochs"],
        "best_val_HR@50": best_val_hr50,
    }

    for split in ("overall", "head", "tail"):
        for key, value in test_metrics[split].items():
            row[f"test_{split}_{key}"] = value

    for key, value in test_metrics["long_tail"].items():
        row[f"test_{key}"] = value

    for key, value in test_metrics["counts"].items():
        row[f"test_count_{key}"] = value

    all_rows.append(row)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metrics_df = pd.DataFrame(all_rows)
metrics_df


## 10. Compact final summary

In [ ]:
def make_movielens_summary_table(
    metrics_df: pd.DataFrame,
    method_name: str = "CDN",
    save_path: str | None = "movielens_cdn_summary.csv",
) -> pd.DataFrame:
    """
    One final table:
        one row = method
        values = mean ± std over seeds
    No head/tail sweep.
    """
    selected_metrics = [
        "test_overall_HR@10",
        "test_overall_NDCG@10",
        "test_overall_HR@50",
        "test_overall_NDCG@50",
        "test_head_HR@50",
        "test_head_NDCG@50",
        "test_tail_HR@50",
        "test_tail_NDCG@50",
        "test_CatalogCoverage@50",
        "test_TailCoverage@50",
        "test_AvgPopularity@50",
        "test_TailRatio@50",
        "test_Personalization@50",
    ]

    row = {
        "method": method_name,
        "num_seeds": metrics_df["seed"].nunique() if "seed" in metrics_df.columns else len(metrics_df),
        "head_fraction": CFG["head_fraction"],
        "head_share": f"{100 * CFG['head_fraction']:.1f}%",
        "num_head_items": int(head_mask.sum()),
        "num_tail_items": int((~head_mask).sum()),
        "test_head_share": f"{100 * float(head_mask[test_i].mean()):.2f}%",
        "test_tail_share": f"{100 * float((~head_mask[test_i]).mean()):.2f}%",
    }

    for hp_col in ["lr", "dropout", "weight_decay", "gamma", "n_experts", "final_epochs"]:
        if hp_col in metrics_df.columns:
            vals = metrics_df[hp_col].dropna().unique()
            if len(vals) == 1:
                row[hp_col] = vals[0]

    if "best_val_HR@50" in metrics_df.columns:
        vals = metrics_df["best_val_HR@50"].dropna().to_numpy(dtype=float)
        if len(vals) > 0:
            mean = float(np.mean(vals))
            std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            row["best_val_HR@50"] = f"{mean:.2f} ± {std:.2f}"

    for metric in selected_metrics:
        if metric not in metrics_df.columns:
            continue
        vals = metrics_df[metric].dropna().to_numpy(dtype=float)
        out_name = metric.replace("test_", "")
        if len(vals) == 0:
            row[out_name] = "nan"
            continue
        mean = float(np.mean(vals))
        std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        if "AvgPopularity" in out_name:
            row[out_name] = f"{mean:.4f} ± {std:.4f}"
        else:
            row[out_name] = f"{mean:.2f} ± {std:.2f}"

    summary_df = pd.DataFrame([row])
    if save_path is not None:
        summary_df.to_csv(save_path, index=False)
        print(f"saved: {save_path}")
    return summary_df


summary_df = make_movielens_summary_table(
    metrics_df=metrics_df,
    method_name="CDN",
    save_path="movielens_cdn_summary.csv",
)

summary_df
